The final submission file (`sub_zindi_farm2feed.csv`) is generated through a multi-stage pipeline that combines regression, classification ensembles, and heuristic post-processing.

Here is the brief step-by-step explanation:

**1. Quantity Prediction (`LB_Fqty.csv`)**

* **Method:** A hybrid approach using **LightGBM Regressor** and statistical heuristics.
* **Logic:**
* **High-Variance Customers ("Blacklist"):** Predicted using the LightGBM model.
* **Stable Customers:** Predicted using a rolling quantile heuristic (35th percentile for 1-week, 20th percentile for 2-week).
* **Outliers:** The specific outlier ID `438_172` is temporarily forced to 0.



**2. Purchase Probability - 1 Week (`sub.csv`)**

* **Method:** An ensemble of two different model runs targeting `Target_purchase_next_1w`.
* **Run 1 (`LB_T1.csv`):** A single LightGBM Classifier model.
* **Run 2 (`LBH.csv`):** A **5-seed ensemble** of LightGBM Classifiers averaged together to reduce noise.
* **Blending:** The final 1-week probability is the average of Run 1 and Run 2: `(LBH + LB_T1) / 2`.

**3. Purchase Probability - 2 Weeks (`LB_FT2.csv`)**

* **Method:** A separate LightGBM Classifier trained specifically on the 2-week target (`Target_qty_next_2w`).
* **Logic:** It uses deep trees (`max_depth=20`) and updates the `Target_purchase_next_2w` column in the existing submission file.

**4. Final part handling outlier (`sub_zindi_farm2feed.csv`)**

* **Method:** Manual override for the specific outlier `438_172` using global statistics.
* **Logic:**
* **1W Quantity:** Overwritten with the global median of the training set.
* **2W Quantity:** Overwritten with a decay sequence (0.6x, 0.2x, etc.) applied to the outlier's historical median.

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import gc
from lightgbm import LGBMRegressor

# ==========================================
# 1. SETUP & DATA LOADING
# ==========================================
print("--- 1. LOADING DATA ---")
train_path = 'Train.csv'
test_path = 'Test.csv'
sub_path = 'SampleSubmission.csv'

train_raw = pl.read_csv(train_path).with_columns(pl.col('week_start').str.to_date(strict=False))
test_raw = pl.read_csv(test_path).with_columns(pl.col('week_start').str.to_date(strict=False))
submission = pd.read_csv(sub_path)

def make_id(df):
    return df.with_columns((pl.col('customer_id').cast(pl.Utf8) + "_" + pl.col('product_unit_variant_id').cast(pl.Utf8)).alias('interaction_id'))

train_raw = make_id(train_raw)
test_raw = make_id(test_raw)

# Segment Identification
OUTLIER_ID = '438_172'
stats_1w = train_raw.group_by('interaction_id').agg(pl.col('Target_qty_next_1w').std().alias('std_1w')).fill_null(0)
blacklist_ids = stats_1w.filter((pl.col('std_1w') > 1) & (pl.col('interaction_id') != OUTLIER_ID))['interaction_id'].to_list()

# ==========================================
# 2. FEATURE FUNCTION (The Formula)
# ==========================================
def build_features(df_in, target_col, gap, is_test=False):
    # Formula: gap // 7 - 1
    shift_rows = (gap // 7) - 1
    df = df_in.with_columns(pl.col(target_col).shift(shift_rows).over('interaction_id').alias('lag'))

    feature_cols = []
    for w in [4, 8, 16]:
        for q in [0.2, 0.35, 0.5]:
            name = f"q{q}_w{w}"
            df = df.with_columns(pl.col('lag').rolling_quantile(window_size=w, quantile=q).over('interaction_id').alias(name))
            feature_cols.append(name)

    if is_test:
        return df.group_by('interaction_id').tail(1).fill_null(0), feature_cols
    return df.drop_nulls(), feature_cols

# ==========================================
# 3. GENERATE TARGET 1W (Calculated Bridge)
# ==========================================
print("--- 2. PROCESSING TARGET 1W ---")
last_date = train_raw['week_start'].max()
bridge_1w = train_raw.filter(pl.col('week_start') == last_date).with_columns([
    (pl.col('week_start') + pl.duration(days=7)).alias('week_start'),
    (pl.col('Target_qty_next_2w') - pl.col('Target_qty_next_1w')).clip(0, 999999).alias('Target_qty_next_1w')
])
hist_1w = pl.concat([train_raw.select(['interaction_id', 'week_start', 'Target_qty_next_1w']),
                     bridge_1w.select(['interaction_id', 'week_start', 'Target_qty_next_1w'])]).sort(['interaction_id', 'week_start'])

preds_1w_list = []
GAPS = [14, 21, 28, 35, 42, 49]
test_dates = sorted(test_raw['week_start'].unique())

for i, gap in enumerate(GAPS):
    t_date = test_dates[i]
    # Blacklist ML
    bl_hist = hist_1w.filter(pl.col('interaction_id').is_in(blacklist_ids))
    train_data, feats = build_features(bl_hist, 'Target_qty_next_1w', gap, is_test=False)
    test_data, _ = build_features(bl_hist, 'Target_qty_next_1w', gap, is_test=True)
    model = LGBMRegressor(objective='mae', n_estimators=1000, learning_rate=0.07, verbose=-1).fit(train_data[feats].to_pandas(), train_data['Target_qty_next_1w'].to_pandas())
    bl_ids = test_raw.filter((pl.col('week_start') == t_date) & (pl.col('interaction_id').is_in(blacklist_ids))).select(['ID', 'interaction_id'])
    bl_p = bl_ids.join(test_data, on='interaction_id', how='left').fill_null(0)
    bl_preds = np.maximum(model.predict(bl_p[feats].to_pandas()), 0)

    # Normal Heuristic (0.35)
    norm_ids = test_raw.filter((pl.col('week_start') == t_date) & (~pl.col('interaction_id').is_in(blacklist_ids)) & (pl.col('interaction_id') != OUTLIER_ID)).select(['ID', 'interaction_id'])
    norm_val = hist_1w.filter(~pl.col('interaction_id').is_in(blacklist_ids)).with_columns(pl.col('Target_qty_next_1w').shift((gap//7)-1).over('interaction_id').alias('lag')).group_by('interaction_id').agg(pl.col('lag').tail(8).quantile(0.35, interpolation='nearest').alias('pred')).fill_null(0)
    norm_p = norm_ids.join(norm_val, on='interaction_id', how='left').fill_null(0)

    # Outlier
    out_ids = test_raw.filter((pl.col('week_start') == t_date) & (pl.col('interaction_id') == OUTLIER_ID)).select(['ID']).with_columns(pl.lit(0.0).alias('pred'))

    preds_1w_list.append(pl.concat([pl.DataFrame({'ID': bl_p['ID'], 'p1': bl_preds}), norm_p.select(['ID', pl.col('pred').alias('p1')]), out_ids.select(['ID', pl.col('pred').alias('p1')])]))

    # [FIX] Delete Loop Variables to Free RAM
    del bl_hist, train_data, test_data, model, bl_p, norm_p, bl_preds
    gc.collect()

# [FIX] Clean up history
del hist_1w
gc.collect()

# ==========================================
# 4. GENERATE TARGET 2W (Zero Bridge)
# ==========================================
print("--- 3. PROCESSING TARGET 2W ---")
bridge_2w = train_raw.filter(pl.col('week_start') == last_date).with_columns([
    (pl.col('week_start') + pl.duration(days=7)).alias('week_start'),
    pl.lit(0.0).cast(pl.Float64).alias('Target_qty_next_2w')
])
hist_2w = pl.concat([train_raw.select(['interaction_id', 'week_start', 'Target_qty_next_2w']),
                     bridge_2w.select(['interaction_id', 'week_start', 'Target_qty_next_2w'])]).sort(['interaction_id', 'week_start'])

preds_2w_list = []
for i, gap in enumerate(GAPS):
    t_date = test_dates[i]
    # Blacklist ML
    bl_hist = hist_2w.filter(pl.col('interaction_id').is_in(blacklist_ids))
    train_data, feats = build_features(bl_hist, 'Target_qty_next_2w', gap, is_test=False)
    test_data, _ = build_features(bl_hist, 'Target_qty_next_2w', gap, is_test=True)
    model = LGBMRegressor(objective='mae', n_estimators=1000, learning_rate=0.07, verbose=-1).fit(train_data[feats].to_pandas(), train_data['Target_qty_next_2w'].to_pandas())
    bl_ids = test_raw.filter((pl.col('week_start') == t_date) & (pl.col('interaction_id').is_in(blacklist_ids))).select(['ID', 'interaction_id'])
    bl_p = bl_ids.join(test_data, on='interaction_id', how='left').fill_null(0)
    bl_preds = np.maximum(model.predict(bl_p[feats].to_pandas()), 0)

    # Normal Heuristic (0.20)
    norm_ids = test_raw.filter((pl.col('week_start') == t_date) & (~pl.col('interaction_id').is_in(blacklist_ids)) & (pl.col('interaction_id') != OUTLIER_ID)).select(['ID', 'interaction_id'])
    norm_val = hist_2w.filter(~pl.col('interaction_id').is_in(blacklist_ids)).with_columns(pl.col('Target_qty_next_2w').shift((gap//7)-1).over('interaction_id').alias('lag')).group_by('interaction_id').agg(pl.col('lag').tail(8).quantile(0.20, interpolation='nearest').alias('pred')).fill_null(0)
    norm_p = norm_ids.join(norm_val, on='interaction_id', how='left').fill_null(0)

    # Outlier
    out_ids = test_raw.filter((pl.col('week_start') == t_date) & (pl.col('interaction_id') == OUTLIER_ID)).select(['ID']).with_columns(pl.lit(0.0).alias('pred'))

    preds_2w_list.append(pl.concat([pl.DataFrame({'ID': bl_p['ID'], 'p2': bl_preds}), norm_p.select(['ID', pl.col('pred').alias('p2')]), out_ids.select(['ID', pl.col('pred').alias('p2')])]))

    # [FIX] Delete Loop Variables to Free RAM
    del bl_hist, train_data, test_data, model, bl_p, norm_p, bl_preds
    gc.collect()

# [FIX] Clean up history
del hist_2w
gc.collect()

# ==========================================
# 5. MERGE & SAVE
# ==========================================
print("--- 4. FINAL SAVING ---")
df1 = pl.concat(preds_1w_list).to_pandas()
df2 = pl.concat(preds_2w_list).to_pandas()

submission = submission.merge(df1, on='ID', how='left').merge(df2, on='ID', how='left')
submission['Target_qty_next_1w'] = submission['p1'].fillna(0)
submission['Target_qty_next_2w'] = submission['p2'].fillna(0)
submission=submission.drop(['p1', 'p2'], axis=1)
submission.to_csv('LB_Fqty.csv', index=False)

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
import gc

warnings.filterwarnings('ignore')

# --- CONFIGURATION ---
TARGET = "Target_purchase_next_1w"
SRC_QTY_1W = "Target_qty_next_1w"
SRC_QTY_2W = "Target_qty_next_2w"
GAPS = [7, 14, 21, 28, 35, 42]
N_ESTIMATORS = 60
LEARNING_RATE = 0.05

# ==========================================
# 1. DATA LOADING
# ==========================================
print(">>> Loading Data for Submission...")
df_train_raw = pl.read_csv('Train.csv', infer_schema_length=10000)
df_test_raw = pl.read_csv('Test.csv', infer_schema_length=10000)
df_cust = pl.read_csv('customer_data.csv', infer_schema_length=10000)
#here we load above prediction made to update current target
df_sub_base = pd.read_csv('LB_Fqty.csv')

# Clean Customer Data
df_cust = df_cust.unique(subset=["customer_id"]).with_columns([
    pl.col("customer_created_at").cast(pl.String).str.slice(0, 10).str.to_date("%Y-%m-%d", strict=False),
    pl.col("customer_id").cast(pl.Int64)
])

def clean_and_filter(df):
    df = df.with_columns(pl.col("week_start").cast(pl.String).str.slice(0, 10).str.to_date("%Y-%m-%d", strict=False))
    df = df.join(df_cust, on="customer_id", how="left")
    df = df.with_columns([
        ((pl.col("week_start").cast(pl.Date) - pl.col("customer_created_at").cast(pl.Date))
         .dt.total_days().fill_null(0)).alias("turnday")
    ])
    return df.filter(pl.col("turnday") >= -500)

df_train = clean_and_filter(df_train_raw)
df_test = clean_and_filter(df_test_raw)

# [FIX] Delete raw inputs immediately
del df_train_raw, df_test_raw, df_cust
gc.collect()

# ==========================================
# 2. BRIDGE CREATION
# ==========================================
print(">>> Creating Bridge Week...")
old_cutoff = df_train["week_start"].max()
df_bridge = df_train.filter(pl.col("week_start") == old_cutoff)
bridge_bin = ((pl.col(SRC_QTY_2W) - pl.col(SRC_QTY_1W)).clip(0, None) > 0.0001).cast(pl.Int64)

df_bridge = df_bridge.with_columns([
    bridge_bin.alias(TARGET),
    (pl.col("week_start") + pl.duration(days=7)).alias("week_start"),
    pl.lit("bridge_week").alias("ID")
])

# ==========================================
# 3. ASSEMBLY
# ==========================================
cols_h = ["ID", "customer_id", "product_unit_variant_id", "customer_category", "customer_status", "week_start", "turnday", TARGET]
df_history = pl.concat([
    df_train.with_columns(pl.lit("train").alias("ID")).select(cols_h),
    df_bridge.select(cols_h),
    df_test.with_columns([pl.lit(0).cast(pl.Int64).alias(TARGET)]).select(cols_h)
]).with_columns([
    pl.concat_str([pl.col("customer_id"), pl.lit("_"), pl.col("product_unit_variant_id")]).alias("Cust_Prod"),
    pl.col("customer_category").fill_null("Unknown"),
    pl.col("customer_status").fill_null("Unknown"),
    pl.concat_str([pl.col("customer_category"), pl.lit("_"), pl.col("customer_status")]).alias("Cat_Stat_Int")
])

# [FIX] Delete intermediate dataframes
# We keep a minimal df_test for the join logic in step 5
df_test = df_test.select(["ID", "customer_id", "product_unit_variant_id", "week_start"])
del df_train, df_bridge, bridge_bin
gc.collect()

new_cutoff = df_history.filter(pl.col("ID") == "bridge_week")["week_start"].max()

# ==========================================
# 4. FEATURE ENGINEERING
# ==========================================
print(">>> Generating Lags & Rolling Windows...")
df_history = df_history.sort(["Cust_Prod", "week_start"])
df_history = df_history.with_columns([pl.col(TARGET).shift(i).over("Cust_Prod").fill_null(0).alias(f"cp_lag_{i}") for i in range(1, 41)])

groups = {
    "user": ["customer_id"],
    "prod": ["product_unit_variant_id"],
    "cat": ["customer_category"],
    "cat_stat": ["Cat_Stat_Int"]
}

for prefix, g_cols in groups.items():
    agg = df_history.group_by(g_cols + ["week_start"]).agg(pl.col(TARGET).sum().alias("tmp_sum")).sort(g_cols + ["week_start"])
    agg = agg.with_columns([pl.col("tmp_sum").shift(i).over(g_cols).fill_null(0).alias(f"{prefix}_lag_{i}") for i in range(1, 41)])
    df_history = df_history.join(agg.drop("tmp_sum"), on=g_cols + ["week_start"], how="left")
    # [FIX] Clean inside loop
    del agg
    gc.collect()

def build_features(df, horizon):
    start_lag = horizon // 7
    all_prefixes = ["cp", "user", "prod", "cat", "cat_stat"]
    exprs = [pl.col("ID"), pl.col("customer_status"), pl.lit(horizon).alias("horizon_days"), pl.col("turnday"), pl.col(TARGET).alias("raw_target")]

    for pref in all_prefixes:
        for k in range(1, 6): exprs.append(pl.col(f"{pref}_lag_{start_lag + k - 1}").alias(f"{pref}_eff_lag_{k}"))
        exprs.append(pl.sum_horizontal([f"{pref}_lag_{i}" for i in range(start_lag, 41)]).alias(f"{pref}_cumsum"))
        for k in range(1, 22): exprs.append(pl.col(f"{pref}_lag_{start_lag + k - 1}").alias(f"{pref}_roll_input_{k}"))

    res = df.select(exprs)
    roll_exprs = []
    for pref in all_prefixes:
        roll_exprs.append(pl.mean_horizontal([f"{pref}_roll_input_{i}" for i in range(1, 5)]).alias(f"{pref}_roll_4"))
        roll_exprs.append(pl.mean_horizontal([f"{pref}_roll_input_{i}" for i in range(1, 13)]).alias(f"{pref}_roll_12"))
        roll_exprs.append(pl.mean_horizontal([f"{pref}_roll_input_{i}" for i in range(1, 22)]).alias(f"{pref}_roll_21"))

    return res.with_columns(roll_exprs).drop([c for c in res.columns if "roll_input" in c])

# ==========================================
# 5. DATASET CONSTRUCTION
# ==========================================
train_chunks = [build_features(df_history.filter(pl.col("ID").is_in(["train", "bridge_week"])), g) for g in GAPS]
df_train_final = pl.concat(train_chunks)

# [FIX] Delete chunks
del train_chunks
gc.collect()

test_chunks = []
for h in GAPS:
    target_date = new_cutoff + pl.duration(days=h)
    # Join with minimal df_test to get correct submission IDs
    chunk = df_history.filter(pl.col("week_start") == target_date).drop("ID").join(
        df_test,
        on=["customer_id", "product_unit_variant_id", "week_start"], how="inner"
    )
    if chunk.height > 0: test_chunks.append(build_features(chunk, h))
df_test_final = pl.concat(test_chunks)

# [FIX] Delete chunks, history, and the minimal test key dataframe
del test_chunks, df_history, df_test
gc.collect()

# ==========================================
# 6. MODEL TRAINING
# ==========================================
features = [c for c in df_train_final.columns if any(x in c for x in ["eff_lag", "roll_", "horizon", "turnday", "cumsum", "customer_status"])]
features = [c for c in features if "target" not in c.lower() and c != "ID"]

X_train = df_train_final.select(features).to_pandas()
y_train = df_train_final["raw_target"].to_numpy()

# [FIX] Extract IDs BEFORE deleting df_test_final
test_ids = df_test_final["ID"].to_list()
X_test = df_test_final.select(features).to_pandas()

# [FIX] Delete Polars objects to free space for LightGBM
del df_train_final, df_test_final
gc.collect()

for df_tmp in [X_train, X_test]:
    df_tmp["horizon_days"] = df_tmp["horizon_days"].astype("category")
    df_tmp["customer_status"] = df_tmp["customer_status"].astype("category")

print(f">>> Training LightGBM with {N_ESTIMATORS} Estimators...")
model = lgb.LGBMClassifier(n_estimators=N_ESTIMATORS, learning_rate=0.05, max_depth=15, num_leaves=90, random_state=42, verbose=-1)
model.fit(X_train, y_train)

# [FIX] Delete training data
del X_train, y_train
gc.collect()

# ==========================================
# 7. SUBMISSION GENERATION
# ==========================================
print(">>> Updating Submission File...")
preds = model.predict_proba(X_test)[:, 1]

# Use the extracted test_ids
df_preds = pd.DataFrame({'ID': test_ids, TARGET: preds})

# Use loc method to update only the specific TARGET column based on ID
df_sub_base.set_index('ID', inplace=True)
df_preds.set_index('ID', inplace=True)

# Surgery Update: Only updates the column defined by TARGET
df_sub_base.loc[df_preds.index, TARGET] = df_preds[TARGET]

# Reset index to restore ID column and save
df_sub_base.reset_index(inplace=True)
df_sub_base.to_csv('LB_T1.csv', index=False)

print("✅ Submission file 'LB_T1.csv' generated successfully.")

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import lightgbm as lgb
import gc
import warnings

warnings.filterwarnings('ignore')

# --- CONFIGURATION ---
TARGET = "Target_purchase_next_1w"
FEAT_SOURCE_2W = "Target_purchase_next_2w"
GAPS = [7, 14, 21, 28, 35, 42]

# Ensemble Settings
SEEDS = [0, 11, 42, 123, 1234]
N_ESTIMATORS = [100, 110, 95, 100, 100]

# ==========================================
# 1. DATA LOADING & CLEANING
# ==========================================
print(">>> Loading Data for Final Submission...")
df_train_raw = pl.read_csv('Train.csv', infer_schema_length=10000)
df_test_raw = pl.read_csv('Test.csv', infer_schema_length=10000)
df_cust = pl.read_csv('customer_data.csv', infer_schema_length=10000)
#here same we laod above prediction file too
df_sub_base = pd.read_csv('LB_T1.csv')
df_sub_base['ID'] = df_sub_base['ID'].astype(str)

df_cust = df_cust.unique(subset=["customer_id"]).with_columns([
    pl.col("customer_created_at").cast(pl.String).str.slice(0, 10).str.to_date("%Y-%m-%d", strict=False),
    pl.col("customer_id").cast(pl.Int64)
])

def clean_and_join(df, cust_df):
    df = df.with_columns([
        pl.col("week_start").cast(pl.String).str.slice(0, 10).str.to_date("%Y-%m-%d", strict=False),
        pl.col("ID").cast(pl.String)
    ])
    df = df.join(cust_df, on="customer_id", how="left")
    return df.with_columns([
        ((pl.col("week_start").cast(pl.Date) - pl.col("customer_created_at").cast(pl.Date))
         .dt.total_days().fill_null(0)).alias("turnday")
    ])

df_train = clean_and_join(df_train_raw, df_cust)
df_test = clean_and_join(df_test_raw, df_cust)

# [FIX] Delete raw inputs immediately
del df_train_raw, df_test_raw, df_cust
gc.collect()

# ==========================================
# 2. BRIDGE & HISTORY ASSEMBLY
# ==========================================
print(">>> Assembling History...")
old_cutoff = df_train["week_start"].max()
df_bridge = df_train.filter(pl.col("week_start") == old_cutoff)

# Logic for Target_purchase_next_1w in bridge week
bridge_bin = ((pl.col("Target_qty_next_2w") - pl.col("Target_qty_next_1w")).clip(0, None) > 0.0001).cast(pl.Int64)

df_bridge = df_bridge.with_columns([
    bridge_bin.alias(TARGET),
    pl.lit(0).cast(pl.Int64).alias(FEAT_SOURCE_2W),
    (pl.col("week_start") + pl.duration(days=7)).alias("week_start"),
    pl.lit("bridge_week").alias("ID")
])

cols_h = ["ID", "customer_id", "product_unit_variant_id", "customer_category", "customer_status", "week_start", "turnday", TARGET, FEAT_SOURCE_2W]

# Vertical Stacking of History
df_history = pl.concat([
    df_train.with_columns([pl.lit("train").alias("ID"), pl.col(FEAT_SOURCE_2W).alias(FEAT_SOURCE_2W)]).select(cols_h),
    df_bridge.select(cols_h),
    df_test.with_columns([pl.lit(0).cast(pl.Int64).alias(TARGET), pl.lit(0).cast(pl.Int64).alias(FEAT_SOURCE_2W)]).select(cols_h)
]).with_columns([
    pl.concat_str([pl.col("customer_id"), pl.lit("_"), pl.col("product_unit_variant_id")]).alias("Cust_Prod"),
    pl.col("customer_category").fill_null("Unknown"),
    pl.col("customer_status").fill_null("Unknown")
])

# [FIX] Delete intermediate dataframes
del df_train, df_test, df_bridge, bridge_bin
gc.collect()

# ==========================================
# 3. FEATURE ENGINEERING (Lags)
# ==========================================
print(">>> Generating Lags (RAM Optimized)...")
df_history = df_history.sort(["Cust_Prod", "week_start"])

for src in [TARGET, FEAT_SOURCE_2W]:
    df_history = df_history.with_columns([
        pl.col(src).shift(i).over("Cust_Prod").fill_null(0).alias(f"{src}_lag_{i}") for i in range(1, 41)
    ])

groups = {"user": ["customer_id"], "prod": ["product_unit_variant_id"]}
for prefix, g_cols in groups.items():
    for src in [TARGET, FEAT_SOURCE_2W]:
        agg = df_history.group_by(g_cols + ["week_start"]).agg(pl.col(src).sum().alias("tmp_sum")).sort(g_cols + ["week_start"])
        agg = agg.with_columns([pl.col("tmp_sum").shift(i).over(g_cols).fill_null(0).alias(f"{prefix}_{src}_lag_{i}") for i in range(1, 41)])
        df_history = df_history.join(agg.drop("tmp_sum"), on=g_cols + ["week_start"], how="left")
        # [FIX] Delete aggregation result inside loop
        del agg
        gc.collect()

# ==========================================
# 4. BUILDER FUNCTION
# ==========================================
def build_features(df, horizon, mode="1w"):
    s_lag = (horizon // 7) + 1 if mode == "2w_lens" else (horizon // 7)
    src_suffix = FEAT_SOURCE_2W if mode == "2w_lens" else TARGET

    exprs = [
        pl.col("ID"), pl.col("customer_status"), pl.lit(horizon).alias("horizon_days"),
        pl.col("turnday"), pl.col(TARGET).alias("raw_target")
    ]
    prefixes = ["cp", "user", "prod"]
    for pref in prefixes:
        base_col = f"{src_suffix}" if pref == "cp" else f"{pref}_{src_suffix}"
        for k in range(1, 12): exprs.append(pl.col(f"{base_col}_lag_{s_lag + k - 1}").alias(f"{pref}_eff_lag_{k}"))
        for k in range(1, 22): exprs.append(pl.col(f"{base_col}_lag_{s_lag + k - 1}").alias(f"{pref}_roll_input_{k}"))
        exprs.append(pl.sum_horizontal([f"{base_col}_lag_{i}" for i in range(s_lag, 41)]).alias(f"{pref}_cumsum"))

    res = df.select(exprs)
    roll_exprs = []
    for pref in prefixes:
        roll_exprs.append(pl.mean_horizontal([f"{pref}_roll_input_{i}" for i in range(1, 5)]).alias(f"{pref}_roll_4"))
        roll_exprs.append(pl.mean_horizontal([f"{pref}_roll_input_{i}" for i in range(1, 13)]).alias(f"{pref}_roll_12"))
    return res.with_columns(roll_exprs).drop([c for c in res.columns if "roll_input" in c])

# ==========================================
# 5. TRAINING DATA PREP
# ==========================================
print(">>> Preparing Training Data (Filtered >= -35)...")

train_pool = df_history.filter(
    (pl.col("ID").is_in(["train", "bridge_week"])) &
    (pl.col("turnday") >= -35)
)

# This is the heavy memory part: Create Stack -> Convert to Pandas -> Delete Stack
df_train_stack = pl.concat([
    pl.concat([build_features(train_pool, g, mode="1w") for g in GAPS]),
    pl.concat([build_features(train_pool, g, mode="2w_lens") for g in GAPS])
]).with_columns(pl.when(pl.col("turnday") >= 600).then(2.33).otherwise(1.0).alias("sample_weight"))

# [FIX] Delete pool source
del train_pool
gc.collect()

feats = [c for c in df_train_stack.columns if any(x in c for x in ["eff_lag", "roll_", "horizon", "turnday", "cumsum", "customer_status"])]
feats = [c for c in feats if "target" not in c.lower() and c != "ID"]

X_tr = df_train_stack.select(feats).to_pandas()
y_tr = df_train_stack["raw_target"].to_numpy()
w_tr = df_train_stack["sample_weight"].to_numpy()
X_tr["customer_status"] = X_tr["customer_status"].fillna("Unknown").astype("category")

# [FIX] Delete heavy Polars stack
del df_train_stack
gc.collect()

# ==========================================
# 6. ENSEMBLE TRAINING & INFERENCE
# ==========================================
print(f">>> Training {len(SEEDS)} Model Ensemble...")

accumulated_preds = {}
new_cutoff = df_history.filter(pl.col("ID") == "bridge_week")["week_start"].max()

for i, (seed, n_est) in enumerate(zip(SEEDS, N_ESTIMATORS)):
    print(f"\n   🔄 Round {i+1}/5 (Seed={seed}, N_Est={n_est})...")

    # 1. Train Model
    model = lgb.LGBMClassifier(
        n_estimators=n_est,
        learning_rate=0.03,
        max_depth=12,
        num_leaves=64,
        random_state=seed,
        verbose=-1
    )
    model.fit(X_tr, y_tr, sample_weight=w_tr)

    # 2. Predict on Test Gaps (Dual View)
    print("      > Predicting Test chunks...")
    for g in GAPS:
        target_date = new_cutoff + pl.duration(days=g)
        chunk = df_history.filter(pl.col("week_start") == target_date)

        if chunk.height == 0: continue

        # View 1
        df_v1 = build_features(chunk, g, mode="1w").select(["ID"] + feats).to_pandas()
        df_v1["customer_status"] = df_v1["customer_status"].fillna("Unknown").astype("category")
        ids = df_v1["ID"].values
        p1 = model.predict_proba(df_v1[feats])[:, 1]

        # [FIX] Delete View 1 immediately
        del df_v1

        # View 2
        df_v2 = build_features(chunk, g, mode="2w_lens").select(feats).to_pandas()
        df_v2["customer_status"] = df_v2["customer_status"].fillna("Unknown").astype("category")
        p2 = model.predict_proba(df_v2[feats])[:, 1]

        # [FIX] Delete View 2 immediately
        del df_v2

        # Average Views
        p_avg = (p1 + p2) / 2

        # Accumulate
        for uid, val in zip(ids, p_avg):
            accumulated_preds[uid] = accumulated_preds.get(uid, 0.0) + val

        # [FIX] Delete loop variables
        del chunk, ids, p1, p2, p_avg
        gc.collect()

    # [FIX] Delete model after iteration
    del model
    gc.collect()

# [FIX] Clean up Training Data & History before saving
del X_tr, y_tr, w_tr, df_history
gc.collect()

# ==========================================
# 7. FINALIZE & SAVE
# ==========================================
print("\n>>> Finalizing Ensemble Predictions...")

# Create DataFrame from dictionary
final_ids = list(accumulated_preds.keys())
final_probs = np.array(list(accumulated_preds.values())) / len(SEEDS)

df_final = pd.DataFrame({'ID': final_ids, 'p': final_probs}).set_index('ID')

# Update Base Submission
df_sub_base.set_index('ID', inplace=True)
df_sub_base.update(df_final.rename(columns={'p': TARGET}))
df_sub_base.reset_index().to_csv('LBH.csv', index=False)

print("\n✅ Ensemble Submission generated: LBH.csv")

In [ ]:
#we load both prediction for Target_qty_next_1w to ensemble then save it
A=pd.read_csv('working/LBH.csv')
B=pd.read_csv('LB_T1.csv')
A['Target_qty_next_1w']=(A['Target_qty_next_1w']+B['Target_qty_next_1w'])/2

In [ ]:
A.to_csv('sub.csv', index=False)

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
import gc
from datetime import date

warnings.filterwarnings('ignore')

# ==========================================
# ⚙️ ULTIMATE TUNING CONFIG
# ==========================================
TUNING_CONFIG = {
    # --- BINARY TARGETS ---
    "cp":       {"seq": 16, "cum_seq": 1, "raw_roll_wins": [4, 8, 14, 21], "cum_roll_wins": [4]},
    "cust":     {"seq": 12, "cum_seq": 1, "raw_roll_wins": [4, 8],          "cum_roll_wins": []},
    "prod":     {"seq": 5,  "cum_seq": 1, "raw_roll_wins": [4, 8],          "cum_roll_wins": []},
    "cat":      {"seq": 5,  "cum_seq": 1, "raw_roll_wins": [4, 8],          "cum_roll_wins": []},
    "stat":     {"seq": 5,  "cum_seq": 1, "raw_roll_wins": [4, 8],          "cum_roll_wins": []},
    "pmat":     {"seq": 5,  "cum_seq": 1, "raw_roll_wins": [4, 8],          "cum_roll_wins": []},

    # --- VALUE LEVELS (Spend & Qty 1-Week) ---
    "cp_spend":   {"seq": 8, "cum_seq": 1, "raw_roll_wins": [4, 8], "cum_roll_wins": []},
    "cp_qty":     {"seq": 8, "cum_seq": 1, "raw_roll_wins": [4, 8], "cum_roll_wins": []},
    "cust_spend": {"seq": 8, "cum_seq": 1, "raw_roll_wins": [4, 8], "cum_roll_wins": []},
    "cust_qty":   {"seq": 8, "cum_seq": 1, "raw_roll_wins": [4, 8], "cum_roll_wins": []},
    "prod_spend": {"seq": 8, "cum_seq": 1, "raw_roll_wins": [4, 8], "cum_roll_wins": []},
    "prod_qty":   {"seq": 8, "cum_seq": 1, "raw_roll_wins": [4, 8], "cum_roll_wins": []},

    # --- SELF-LAGS (Target_qty_next_2w History) ---
    "cp_q2w":     {"seq": 8, "cum_seq": 1, "raw_roll_wins": [4, 8], "cum_roll_wins": []},
    "cust_q2w":   {"seq": 8, "cum_seq": 1, "raw_roll_wins": [4, 8], "cum_roll_wins": []},
    "prod_q2w":   {"seq": 8, "cum_seq": 1, "raw_roll_wins": [4, 8], "cum_roll_wins": []},
}

# ==========================================
# 🛠️ FEATURE BUILDER
# ==========================================
def build_features(df, target_gap, anchor_dt=None, mode="train"):
    start_lag = target_gap // 7

    if mode == "test" and anchor_dt is not None:
        actual_date = df["week_start"][0]
        gap_days = (pd.to_datetime(actual_date).date() - anchor_dt).days
        print(f"      [TEST] Horizon: {target_gap}d | Actual Gap: {gap_days}d | Rows: {df.height}")

    exprs = [
        pl.col("ID"),
        pl.lit(target_gap).cast(pl.Int16).alias("horizon"),
        pl.col("target_binary"),
        pl.col("turnday"),
        pl.col("day_of_month"),
        pl.col("customer_status").cast(pl.String).cast(pl.Categorical),
    ]

    # --- Zero Streak (Recency) ---
    streak_expr = pl.when(pl.col(f"cp_l_{start_lag}") == 1).then(0)
    for i in range(1, 13):
        lag_idx = start_lag + i
        streak_expr = streak_expr.when(pl.col(f"cp_l_{lag_idx}") == 1).then(i)
    streak_expr = streak_expr.otherwise(13).alias("cp_zero_streak")
    exprs.append(streak_expr)

    for lvl, settings in TUNING_CONFIG.items():
        for k in range(settings["seq"]):
            exprs.append(pl.col(f"{lvl}_l_{start_lag + k}").alias(f"{lvl}_raw_l_{k+1}"))
        for k in range(settings["cum_seq"]):
            exprs.append(pl.col(f"{lvl}_c_{start_lag + k}").alias(f"{lvl}_cum_l_{k+1}"))
        for w in settings["raw_roll_wins"]:
            exprs.append(pl.mean_horizontal([f"{lvl}_l_{i}" for i in range(start_lag, start_lag + w)]).alias(f"{lvl}_raw_roll_{w}"))
        for w in settings["cum_roll_wins"]:
            exprs.append(pl.mean_horizontal([f"{lvl}_c_{i}" for i in range(start_lag, start_lag + w)]).alias(f"{lvl}_cum_roll_{w}"))

    return df.select(exprs)

def apply_lags_and_cum_pools(df):
    POOL_SOURCES = {
        # --- Standard Binary Pools ---
        "cp":       {"keys": ["Cust_Prod"],                "val": "target_binary",         "agg": "max"},
        "cust":     {"keys": ["customer_id"],              "val": "target_binary",         "agg": "sum"},
        "prod":     {"keys": ["product_unit_variant_id"], "val": "target_binary",         "agg": "sum"},
        "cat":      {"keys": ["customer_category"],        "val": "target_binary",         "agg": "sum"},
        "stat":     {"keys": ["customer_status"],          "val": "target_binary",         "agg": "sum"},
        "pmat":     {"keys": ["Prod_Mat"],                 "val": "target_binary",         "agg": "sum"},

        # --- Value Pools ---
        "cp_spend":   {"keys": ["Cust_Prod"],                "val": "Target_spend_next_1w", "agg": "sum"},
        "cp_qty":     {"keys": ["Cust_Prod"],                "val": "Target_qty_next_1w",   "agg": "sum"},
        "cust_spend": {"keys": ["customer_id"],              "val": "Target_spend_next_1w", "agg": "sum"},
        "cust_qty":   {"keys": ["customer_id"],              "val": "Target_qty_next_1w",   "agg": "sum"},
        "prod_spend": {"keys": ["product_unit_variant_id"], "val": "Target_spend_next_1w", "agg": "sum"},
        "prod_qty":   {"keys": ["product_unit_variant_id"], "val": "Target_qty_next_1w",   "agg": "sum"},

        # --- Self-Lags (2-Week Qty) ---
        "cp_q2w":     {"keys": ["Cust_Prod"],                "val": "Target_qty_next_2w",   "agg": "sum"},
        "cust_q2w":   {"keys": ["customer_id"],              "val": "Target_qty_next_2w",   "agg": "sum"},
        "prod_q2w":   {"keys": ["product_unit_variant_id"], "val": "Target_qty_next_2w",   "agg": "sum"},
    }

    MAX_LAG = 35

    for prefix, config in POOL_SOURCES.items():
        keys = config["keys"]
        val_col = config["val"]
        agg_method = config["agg"]

        print(f"   -> Building {prefix} Pools (Src: {val_col})...")
        dtype = pl.Float32 if "spend" in prefix or "qty" in prefix or "q2w" in prefix else pl.Float32

        if agg_method == "max":
            agg_op = pl.col(val_col).max()
        else:
            agg_op = pl.col(val_col).sum()

        agg = df.group_by(keys + ["week_start"]).agg(agg_op.alias("v")).sort(keys + ["week_start"])
        agg = agg.with_columns(pl.col("v").cum_sum().over(keys).cast(dtype).alias("vc"))

        lag_exprs = []
        for i in range(1, MAX_LAG + 1):
            lag_exprs.append(pl.col("v").shift(i).over(keys).fill_null(0).cast(dtype).alias(f"{prefix}_l_{i}"))
            lag_exprs.append(pl.col("vc").shift(i).over(keys).fill_null(0).cast(dtype).alias(f"{prefix}_c_{i}"))

        agg = agg.with_columns(lag_exprs)

        # Memory-safe Join
        df = df.lazy().join(agg.lazy().drop(["v", "vc"]), on=keys + ["week_start"], how="left").collect()

        # [FIX] Clean up loop variable immediately
        del agg
        gc.collect()

    return df

def prepare_data(df_tr, df_test):
    df_tr = df_tr.with_columns(pl.col("week_start").cast(pl.String).str.slice(0, 10).str.to_date(strict=False))
    df_test = df_test.with_columns(pl.col("week_start").cast(pl.String).str.slice(0, 10).str.to_date(strict=False))

    tr_max_dt = df_tr["week_start"].max()

    def finalize_set(df, source_tag):
        if "selling_price" in df.columns:
            df = df.drop("selling_price")

        target = (pl.col("Target_qty_next_2w").fill_null(0) > 0).cast(pl.Int8) if "Target_qty_next_2w" in df.columns else pl.lit(0).cast(pl.Int8)

        spend = pl.col("Target_spend_next_1w").fill_null(0).cast(pl.Float32) if "Target_spend_next_1w" in df.columns else pl.lit(0.0).cast(pl.Float32)
        qty   = pl.col("Target_qty_next_1w").fill_null(0).cast(pl.Float32)   if "Target_qty_next_1w" in df.columns else pl.lit(0.0).cast(pl.Float32)
        qty2w = pl.col("Target_qty_next_2w").fill_null(0).cast(pl.Float32)   if "Target_qty_next_2w" in df.columns else pl.lit(0.0).cast(pl.Float32)

        return df.with_columns([
            target.alias("target_binary"),
            spend.alias("Target_spend_next_1w"),
            qty.alias("Target_qty_next_1w"),
            qty2w.alias("Target_qty_next_2w"),
            pl.lit(source_tag).alias("source")
        ])

    df_tr = finalize_set(df_tr, "train")
    df_test = finalize_set(df_test, "test")

    df_bridge = df_tr.filter(pl.col("week_start") == tr_max_dt).with_columns([
        pl.lit(0).cast(pl.Int8).alias("target_binary"),
        pl.lit(0.0).cast(pl.Float32).alias("Target_spend_next_1w"),
        pl.lit(0.0).cast(pl.Float32).alias("Target_qty_next_1w"),
        pl.lit(0.0).cast(pl.Float32).alias("Target_qty_next_2w"),
        (pl.col("week_start") + pl.duration(days=7)).alias("week_start"),
        pl.lit("bridge").alias("source")
    ])

    df = pl.concat([df_tr, df_bridge, df_test], how="diagonal")

    # [FIX] Ensure customer data components are handled if present, else fallback safely
    if "customer_created_at" in df.columns:
        df = df.with_columns([
            pl.col("customer_created_at").cast(pl.String).str.slice(0, 10).str.to_date(strict=False).fill_null(pl.col("week_start")),
        ])
    else:
        # Fallback if customer data wasn't joined: assume turn day is 0 or needs to be handled
        print("Warning: customer_created_at not found. Defaulting turnday.")
        df = df.with_columns(pl.col("week_start").alias("customer_created_at"))

    df = df.with_columns([
        pl.concat_str([pl.col("customer_id"), pl.lit("_"), pl.col("product_unit_variant_id")]).alias("Cust_Prod"),
        pl.col("customer_category").fill_null("Unknown"),
        pl.col("customer_status").fill_null("Unknown"),
    ]).with_columns([
        ((pl.col("week_start") - pl.col("customer_created_at")).dt.total_days().cast(pl.Int16).alias("turnday")),
        pl.col("week_start").dt.day().cast(pl.Int8).alias("day_of_month")
    ])

    df = df.with_columns(
        (pl.col("turnday") > 100).cast(pl.Int8).alias("is_mature")
    ).with_columns(
        pl.concat_str([
            pl.col("product_unit_variant_id"),
            pl.lit("_"),
            pl.col("is_mature")
        ]).alias("Prod_Mat")
    )

    return df

# ==========================================
# 📊 EXECUTION (SUBMISSION)
# ==========================================
print(">>> Loading Data...")
df_tr_raw = pl.read_csv('Train.csv', infer_schema_length=10000)
df_test_raw = pl.read_csv('Test.csv', infer_schema_length=10000)

# [FIX] Load Customer Data and Join (Missing in original snippet but required for turnday)
try:
    df_cust = pl.read_csv('customer_data.csv', infer_schema_length=10000)
    df_cust = df_cust.unique(subset=["customer_id"]).with_columns([
        pl.col("customer_created_at").cast(pl.String).str.slice(0, 10).str.to_date("%Y-%m-%d", strict=False),
        pl.col("customer_id").cast(pl.Int64)
    ])
    df_tr_raw = df_tr_raw.join(df_cust, on="customer_id", how="left")
    df_test_raw = df_test_raw.join(df_cust, on="customer_id", how="left")
    del df_cust
    gc.collect()
except Exception as e:
    print(f"Note: Could not load customer_data ({e}). Proceeding with raw files.")

tr_anchor = df_tr_raw.with_columns(pl.col("week_start").cast(pl.String).str.slice(0, 10).str.to_date(strict=False))["week_start"].max()
print(f"Training Anchor Date: {tr_anchor}")

# [FIX] Load Submission Base here
df_sub_base = pd.read_csv('LB_Fqty.csv')

df_hist = prepare_data(df_tr_raw, df_test_raw)

# [FIX] Delete raw files
del df_tr_raw, df_test_raw
gc.collect()

df_hist = apply_lags_and_cum_pools(df_hist)

print("\n>>> BUILDING STACKS...")
# [FIX] Build Train Stack, Convert to Pandas, then Delete Polars Object
train_final = pl.concat([
    build_features(df_hist.filter((pl.col("source") == "train") & (pl.col("turnday") > -35)), g, None, "train")
    for g in [14, 21, 28, 35, 42, 49]
])

feat_cols = [c for c in train_final.columns if c not in ["ID", "target_binary"]]
X_tr = train_final.select(feat_cols).to_pandas()
y_tr = train_final["target_binary"].to_numpy()

# [FIX] Delete Train Stack in Polars
del train_final
gc.collect()

# [FIX] Build Test Stack
test_list = []
for g in [14, 21, 28, 35, 42, 49]:
    exp_week = (pd.to_datetime(tr_anchor) + pd.Timedelta(days=g)).date()
    subset = df_hist.filter(
        (pl.col("source") == "test") &
        (pl.col("week_start") == exp_week) &
        (pl.col("turnday") > -35)
    )
    if subset.height > 0:
        test_list.append(build_features(subset, g, tr_anchor, "test"))

if len(test_list) > 0:
    test_final = pl.concat(test_list)
else:
    raise ValueError("❌ No Test rows aligned. Check dates!")

# [FIX] Delete History and list
del df_hist, test_list
gc.collect()

# [FIX] Convert Test to Pandas and delete Polars
X_test = test_final.select(feat_cols).to_pandas()
test_ids = test_final["ID"].to_list() # Extract IDs before deleting

del test_final
gc.collect()

# --- MODELING ---
for df_tmp in [X_tr, X_test]:
    df_tmp["horizon"] = df_tmp["horizon"].astype("category")
    df_tmp["customer_status"] = df_tmp["customer_status"].astype("category")

print(f"\n>>> Training Final Model on {len(X_tr)} rows with {len(feat_cols)} features...")
# 🔥 FINAL MODEL CONFIG: n_estimators = 145
model = lgb.LGBMClassifier(n_estimators=230, learning_rate=0.03, max_depth=20, max_leaves=64, random_state=42, verbose=-1)

model.fit(X_tr, y_tr)

# [FIX] Delete training data
del X_tr, y_tr
gc.collect()

# --- PREDICTION ---
print("\n>>> Predicting on Test Set...")
preds = model.predict_proba(X_test)[:, 1]

df_preds = pd.DataFrame({
    "ID": test_ids,
    "prob": preds
})

# --- SUBMISSION UPDATE ---
print("\n>>> Updating Submission File...")
#we load firts last prediction for almost three targte now remian this current target only
# this sub.csv file is lastly save file after ensembled
sub_path = 'sub.csv'
sub = pd.read_csv(sub_path)

sub.set_index('ID', inplace=True)
df_preds.set_index('ID', inplace=True)

sub.update(df_preds.rename(columns={'prob': 'Target_purchase_next_2w'}))
sub.reset_index(inplace=True)
sub['Target_purchase_next_2w'] = sub['Target_purchase_next_2w'].fillna(0)

sub.to_csv("LB_FT2.csv", index=False)
print(f"✅ Submission Saved! Updated rows based on ID match.")

In [ ]:
import polars as pl
import pandas as pd
import numpy as np

# ==========================================
# 1. SETUP & PATHS
# ==========================================
print("--- 1. LOADING DATA ---")
train_path = 'Train.csv'
test_path = 'Test.csv'

#we load submission file that made prediction for all column aboveto update outlier only
sub_path = 'LB_FT2.csv'

# Load Train for Median calculation
train = pl.read_csv(train_path).with_columns([
    (pl.col('customer_id').cast(pl.Utf8) + "_" + pl.col('product_unit_variant_id').cast(pl.Utf8)).alias('interaction_id')
])

# Load Test and Submission (Using Pandas for the .loc operations)
test_df = pl.read_csv(test_path).with_columns([
    (pl.col('customer_id').cast(pl.Utf8) + "_" + pl.col('product_unit_variant_id').cast(pl.Utf8)).alias('interaction_id')
]).to_pandas()

submission = pd.read_csv(sub_path)

# ==========================================
# 2. CALCULATION CONSTANTS
# ==========================================
OUTLIER_ID = '438_172'

# A. Target 1W: Global Median
global_median_1w = train.select(pl.col('Target_qty_next_1w')).median().item()

# B. Target 2W: Outlier Decay Logic
outlier_hist = train.filter(pl.col('interaction_id') == OUTLIER_ID)
outlier_median_2w = outlier_hist.select(pl.col('Target_qty_next_2w').tail(4)).median().item()
multipliers_2w = [0.6, 0.01, 0.001, 0.0001, 0.00001, 0.000001]
decay_values_2w = [outlier_median_2w * m for m in multipliers_2w]

print(f"Global 1W Median: {global_median_1w}")
print(f"Outlier 2W Base: {outlier_median_2w} | Decay: {decay_values_2w}")

# ==========================================
# 3. SELECTIVE UPDATE USING .LOC
# ==========================================
print(f"--- 2. UPDATING OUTLIER {OUTLIER_ID} ---")

# Step 1: Identify the weeks for the outlier in the test set
outlier_test_dates = sorted(test_df[test_df['interaction_id'] == OUTLIER_ID]['week_start'].unique())

for i, date in enumerate(outlier_test_dates):
    # 1W Update: Global Median for the outlier rows
    # 2W Update: Decay Value for the outlier rows

    # Create the specific mask for this ID and this Week
    mask = (submission['ID'].str.contains(OUTLIER_ID)) & (test_df['week_start'] == date)

    # Update only Target_qty_next_1w
    submission.loc[mask, 'Target_qty_next_1w'] = global_median_1w

    # Update only Target_qty_next_2w with the decay sequence
    if i < len(decay_values_2w):
        submission.loc[mask, 'Target_qty_next_2w'] = decay_values_2w[i]

# ==========================================
# 4. SAVE
# ==========================================
submission.to_csv('sub_zindi_farm2feed.csv', index=False)

In [ ]:
submission[submission['ID'].str.contains(OUTLIER_ID)]

- The submission code is start from just sample submission then update each target column step by step so when try this code please use step by step and replace when made prediction replace with correct path and if any thing could make problem please let me know thank you. -
- If open to collabartion on later proccess i am ready help all thing like model deployment or building apps or site for this .